# Argus — Dataset Creation (LSTM, windowed)

This notebook builds the **windowed** dataset used to train the LSTM sequence model
(`03_model_training_lstm.ipynb`). It is one of two dataset-creation notebooks:

* **This notebook** — multi-duration sliding windows (1-6s) of per-frame features, each padded
  to a single fixed length and written as one row of a consolidated CSV. Use this for the LSTM.
* [`02_dataset_creation_flat.ipynb`](./02_dataset_creation_flat.ipynb) — one row per single valid
  frame, no windowing. Use this for the RandomForest, the Dense feedforward NN, and (via a
  separate face-crop notebook) the CNN.

Both notebooks share the same raw video source (`dataset/raw_videos_binary/`), the same MediaPipe
feature-extraction pipeline, and the same 2-class label (`Not Drowsy` / `Drowsy`); see "Pipeline
Configuration Constants" below. Keep those constants identical across both notebooks; they are
intentionally redefined in each (Colab-style, no shared imports) rather than centralized.


## Project Overview

**Project:** System for detecting drowsiness in truck drivers and providing preventive assistance.

**Objective:** Define a reproducible pipeline that converts raw `.mp4` recordings into a
structured `.csv` dataset, used to train the Argus `LSTM` (recurrent neural network) to predict
a driver's drowsiness level.

**Solution:** After evaluating several architectures, we chose an `LSTM` because it is
lightweight, easy to implement, and fits our pipeline well: a pretrained `CNN-based` model
(`MediaPipe FaceLandmarker`) extracts facial landmarks, blendshape scores, and head pose per
frame; these are preprocessed into a set of interpretable ratios and scores (see below), which
become the time-series input to the `LSTM` that predicts the drowsiness level.



## Drowsiness levels (labels)


## Drowsiness labels

Argus classifies **two states -- `Not Drowsy` and `Drowsy`**, grounded in the Karolinska
Sleepiness Scale (KSS) partition [UTA-RLDD](https://sites.google.com/view/utarldd/home)
(Ghoddoosian, Galib & Athitsos, 2019) uses, so that dataset's clips can be adopted directly:

| Class | KSS basis | Numeric label |
|---|---|---|
| Not Drowsy | KSS 1-3 (fully alert) and 6-7 (subtle sleepiness, no real effort to stay awake) | 1 |
| Drowsy | KSS 8-9 (actively fighting to stay awake) | 2 |

(KSS 4-5 is UTA-RLDD's deliberately-excluded ambiguous middle zone.)

The raw clips arrive already labelled this way -- each filename encodes `level_1` or `level_2`
(`level_<L>_clip_<N>.mp4`), parsed by regex in the "Video Processing Loop" below. `map_level`
only validates that number; there is no remapping step.

**Data-collection caveat, to state plainly in the titulacion report:** part of the `Drowsy`
footage was *acted* -- performed from a genuine late-stage-fatigue state rather than a fully
alert baseline, since safely filming a real microsleep episode isn't practical. The non-drowsy
footage was self-recorded under genuine fatigue in late-night sessions. `Drowsy` being one of
only two classes makes this caveat more load-bearing, not less.

### Data Collection Methodology

Two sources feed the raw dataset. Both land in `dataset/raw_videos/` on the 3-class
`level_<1-3>` convention; `relabel_binary_raw_videos.ipynb` then collapses that into the
2-class `dataset/raw_videos_binary/` tree (`level_1` = Not Drowsy, `level_2` = Drowsy) that
this notebook actually reads.

**Argus's own footage** (6 subjects) — self-recorded under genuine fatigue: late at night,
after sleep deprivation, not staged. The most severe footage, now labelled `Drowsy`, was
*acted* — but performed from an already-fatigued state rather than a fully alert baseline,
since safely filming a real microsleep episode isn't practical. That `Drowsy` material
therefore has weaker ground-truth confidence than the rest — standard practice in this field,
but state it plainly in the report, the more so now that `Drowsy` is one of only two classes.

**[UTA-RLDD](https://sites.google.com/view/utarldd/home)** (Ghoddoosian, Galib & Athitsos,
CVPRW 2019) — 60 participants, 180 videos (~10 minutes each), each self-recorded by the
participant *while* in a given Karolinska Sleepiness Scale state, reported on UTA-RLDD's
3-way KSS partition (1–3 / 6–7 / 8–9). Argus folds KSS 1–3 and 6–7 → `Not Drowsy`, KSS 8–9 →
`Drowsy` (see "Drowsiness labels" above). Added mainly to raise the subject count: 6 subjects
is far too few for the model to learn anything beyond memorising those specific people, and
left some classes present in only one subject's footage — making a leakage-free,
subject-grouped split with full class coverage on both sides impossible.

Because each UTA-RLDD video carries one constant label for its full ~10 minutes, cutting it
into several shorter non-overlapping sub-clips and giving each the source video's label is
valid. That cutting is done by `src/cv-argus/scripts/extract_uta_rldd_clips.py` (a local
script, not a cell here), which writes the 3-class `level_<1-3>` clips that
`relabel_binary_raw_videos.ipynb` folds down.

## Feature extraction from facial landmarks



We use MediaPipe's pretrained `FaceLandmarker` model, which internally runs a face detector,
a face mesh model, and a blendshape prediction model to output, per frame:

- 478 3D facial landmarks (x, y, z)
- 52 ARKit-style blendshape scores (0–1), pre-normalized for pose and scale
- A 4×4 head-pose transformation matrix

Rather than feeding raw landmark coordinates directly into the LSTM (noisy, pose- and
scale-dependent, and very high-dimensional), we derive a curated set of interpretable features
from these outputs:



1. **Manual EAR / MAR** (`output_face_landmarks`): Eye Aspect Ratio and Mouth Aspect Ratio,
   computed directly from the 2D landmark coordinates using the standard six-point (eyes) and
   four-point (mouth) formulas. We keep these even though blendshapes cover similar ground,
   because they are simple, fully interpretable, and give us an independent baseline to compare
   against the blendshape-based features during the statistical validation step.
2. **Face blendshapes** (`output_face_blendshapes=True`): 52 ARKit-style coefficients
   (`eyeBlinkLeft`, `eyeBlinkRight`, `eyeSquintLeft/Right`, `jawOpen`, `browDownLeft/Right`,
   `mouthClose`, etc.), already normalized by MediaPipe's model for head pose and face scale —
   effectively pre-computed facial action units, and generally less noisy than EAR/MAR when the
   driver's head is turned.
3. **Head pose** (`output_facial_transformation_matrixes=True`): a 4×4 transformation matrix
   from which we extract pitch, yaw, and roll directly. Head nodding is a strong signal of
   microsleep, and is not captured by the blendshapes above.

### Candidate per-frame features


The table below lists every per-frame feature we currently extract as a candidate for the LSTM
input sequence, along with the rationale for including it.

| Category | Feature(s) | Source | Rationale |
|---|---|---|---|
| Eyes (manual EAR) | `EAR_left`, `EAR_right` | 2D landmarks, computed manually | Kept for interpretability and as a comparison baseline, even though it is pose-sensitive |
| Eyes (blendshape) | `eyeBlinkLeft`, `eyeBlinkRight` | Blendshape | Pose-robust equivalent of the same phenomenon captured by EAR |
| Eyes (blendshape) | `eyeSquintLeft`, `eyeSquintRight` | Blendshape | Squinting associated with visual fatigue |
| Eyes (blendshape) | `eyeWideLeft`, `eyeWideRight` | Blendshape | Widening of the eyes as a conscious effort to stay awake (sometimes precedes a lapse) |
| Gaze | `eyeLookDownLeft`, `eyeLookDownRight` | Blendshape | Gaze drifting downward together with the head |
| Mouth (manual MAR) | `MAR` | 2D landmarks, computed manually | Kept for interpretability, same rationale as EAR |
| Mouth / jaw (blendshape) | `jawOpen`, `mouthFunnel`, `mouthStretchLeft`, `mouthStretchRight` | Blendshape | Yawning — **note the speech/singing confound**, see below |
| Mouth (blendshape) | `mouthClose`, `mouthPressLeft`, `mouthPressRight` | Blendshape | Loss of muscle tone around the mouth at rest |
| Eyebrows (blendshape) | `browDownLeft`, `browDownRight`, `browInnerUp` | Blendshape | Conscious effort to stay alert |
| Speech confound flag | `mouthSmileLeft`, `mouthSmileRight`, short-window mouth variance | Blendshape | Auxiliary signal to help detect whether the driver is talking or singing, so it is not mistaken for yawning — relevant since truck drivers frequently talk on the radio/phone or sing |
| Head pose | `pitch`, `yaw`, `roll` | Facial transformation matrix (not a blendshape) | Head nodding is a strong signal of microsleep |
| Sample quality | `detection_confidence`, `frame_has_face` (0/1) | Detector metadata | Used to discard or down-weight low-confidence frames |



### How we will validate which features actually matter




Rather than assuming all of the above are equally useful, each feature's relevance is validated
statistically in [`02_dataset_creation_flat.ipynb`](./02_dataset_creation_flat.ipynb) — which
builds the flat per-frame dataset and runs the analysis on it directly, **not in this notebook**:

1. **Kruskal-Wallis test** (equivalently Mann-Whitney U, since the label has two classes)
   comparing each feature's distribution across the two classes — a low p-value indicates the
   feature discriminates between them.
2. **Spearman correlation** between each feature and the label — a rank-biserial correlation
   with the 0/1 label.
3. **Logistic-regression baseline** on the label — its standardized coefficients give a
   *multivariate* feature-importance ranking (versus the univariate one above), and its
   accuracy / ROC-AUC is a linear-separability check.
4. **Pairwise Spearman correlation matrix between features**, to detect redundancy (e.g. if
   `EAR_left` and `eyeBlinkLeft` are highly correlated, only one may be needed in the final
   input set).

Features that show no significant relationship with the label there are candidates for removal
before finalizing the LSTM input set.


### From thousands of `.npy` files to one padded CSV

Earlier versions of this notebook saved every valid window as its own `.npy` file
(`subject_features_w{duration}s_start{n}.npy`), which meant tens of thousands of tiny binary
files in Drive for a full sliding-window sweep — slow to generate, slow to sync, and awkward to
resume. We now do two things differently:

* **One CSV, not one file per window.** Each window is still generated exactly as before (see
  "Sliding Window Architecture" below), but instead of `np.save`-ing it separately, its feature
  matrix is flattened into a single row and appended to one `lstm_windows.csv` under
  `dataset_processed/`. `metadata.csv` (subject/level/window_duration/dropped_frames) becomes
  just the leading columns of that same CSV rather than a separate index file pointing at
  external `.npy` paths.
* **Fixed-length padding at generation time, not load time.** Every window — whether it came
  from the 1s or the 6s duration bucket — is zero-*pre*-padded up to a single fixed
  `MAX_TIMESTEPS = 60` (see "Pipeline Configuration Constants") before being written. This used
  to be 120 — double `max_context_sec = 6.0` @ `sampling_fps = 10`, kept as unused headroom for
  longer-context experiments later rather than something a duration analysis required — but a
  full extraction run at 120 OOM-crashed the Colab kernel: every window's flattened row is
  `MAX_TIMESTEPS * num_features` wide, and the entire dataset is accumulated as one Python list
  of these rows before a single `pd.DataFrame(rows)` call builds `lstm_windows.csv` at the end,
  so peak RAM scales directly with row width times total window count. `MAX_TIMESTEPS = 60`
  removes that headroom entirely (exactly `max_context_sec * sampling_fps`, zero slack) but
  still fits every window this pipeline actually generates (1-6s), and roughly halves peak
  memory for the same run. `03_model_training_lstm.ipynb` then reshapes each CSV row straight
  back into `(60, num_features)` — no more padding step at
  load time, since the CSV already stores the padded shape.

Padding is still applied **before** the real frames, not after — this must keep matching the
deployment buffer's fill order (`feature_buffer[:, 1:, :]` shifted left, new frame appended at
the end), so a partially-filled buffer at inference time always looks like "zeros first, real
data last", exactly like these padded training rows. Getting this backwards trains the model on
a padding shape that never occurs in production.

Multiple window durations (1-6s, 1s stride) are still generated per clip, same as before —
that's deliberate: it's what lets the LSTM produce a prediction from just a few seconds of
buffered frames rather than only once the buffer is completely full.


# Project Structure & Feature Extraction Strategy



To balance thorough statistical feature analysis with real-time inference constraints, we adopt a sliding window strategy:

* **Sampling Rate:** Videos are processed at a fixed **10 FPS** (`sampling_fps = 10`) to match the expected performance of target hardware.
* **Sliding Window Architecture:** We generate dynamic sequences with durations of **1.0, 2.0, 3.0, 4.0, 5.0, and 6.0 seconds**. These windows slide across each video with a **1-second stride**.
* **Data Integrity & Filtering:** If a face is lost in a frame, a dummy vector is recorded. To ensure the LSTM only trains on continuous, high-quality data, **any window containing even one invalid frame is discarded**.
* **Fixed-Length Padding:** Every surviving window, regardless of its real duration, is zero-*pre*-padded up to `MAX_TIMESTEPS = 60` frames (exactly `max_context_sec * sampling_fps`, no extra headroom — see the "From thousands of `.npy` files to one padded CSV" note above for why) before being stored.
* **Output Artifact:** All padded windows are written as rows of a single `lstm_windows.csv` under `dataset_processed/` — no more one `.npy` file per window.


Here is the complete project folder layout for Argus. It is designed to keep everything organized in Google Drive: `models/` stores our core assets and trained weights, `dataset/raw_videos_binary/` holds our source recordings organized by subject, and `dataset_processed/` houses `lstm_windows.csv` (this notebook's output — one row per padded window) alongside `frame_features.csv` (the companion flat dataset from `02_dataset_creation_flat.ipynb`).



```Text
/content/drive/MyDrive/Argus/
├── models/                    <- Stores our pretrained MediaPipe task & final trained LSTM model weights
├── dataset/
│   └── raw_videos_binary/     <- 2-class tree from relabel_binary_raw_videos.ipynb (level_1=Not Drowsy, level_2=Drowsy)
│       ├── subject_01/
│       │   ├── level_1_clip_01.mp4
│       │   ├── level_1_clip_02.mp4
│       │   ├── level_2_clip_01.mp4
│       │   └── ...
│       ├── subject_02/
│       │   └── ...
│       └── ...
└── dataset_processed/
    ├── lstm_windows.csv       <- Written by this notebook: one row per padded (60-timestep) window
    └── frame_features.csv     <- Written by 02_dataset_creation_flat.ipynb: one row per valid frame
```
**Key Folder Breakdown**

*    `models/`: Dedicated home for all model files—housing both the pretrained
* `dataset/raw_videos_binary/`: The raw input directory structured cleanly by subject ID (e.g., `subject_01/`, `subject_02/`), containing the original .mp4 video clips.

*    `dataset_processed/`: Holds `lstm_windows.csv` (this notebook's output) and `frame_features.csv` (from the companion flat dataset-creation notebook). This entire folder can be compressed into a `.zip` archive whenever you need a fast backup or want to move your processed dataset across different machine learning environments.

## Google Drive Connection

We mount Google Drive to seamlessly save and load our project files (pretrained models, raw video datasets, and processed artifacts) directly in the cloud, ensuring that all our work persists safely across Google Colab session disconnects.

In [ ]:
from google.colab import drive
import os

# Force TensorFlow to never see a GPU in this process. This notebook's pipeline (OpenCV video
# decode + MediaPipe CPU inference + GeometricRatioFeatureLayer's cheap ratio math) never needs
# one, and if a GPU accelerator is attached, letting TF initialize a CUDA context here before
# 01's parallel-extraction ProcessPoolExecutor forks worker processes is a common cause of
# "BrokenProcessPool: ... terminated abruptly" -- CUDA contexts don't survive fork. Setting this
# before TF is ever imported avoids that entire class of crash regardless of runtime/accelerator.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# Mount Google Drive in the Colab environment
drive.mount("/content/drive")

# Define the base root path for the Argus project
project_folder = "/content/drive/MyDrive/Argus"

print(f"Google Drive successfully mounted! Base project directory: {project_folder}")


## Project Directory Setup & Validation
Firstly, validate and initialize the project root folder.

In [ ]:
import os

if not os.path.exists(project_folder):
    raise Exception(f"The target directory could not be initialized: '{project_folder}'.")

Then, validate and create the necessary project folders to ensure they exist.


In [ ]:
# Drowsiness class labels. The raw clips arrive already binary -- each filename encodes
#   level_1 = "Not Drowsy"  (fully-alert + low-vigilant footage)
#   level_2 = "Drowsy"
# map_level (Pipeline Configuration Constants cell below) only validates that number; there
# is no class-remapping step anywhere in this pipeline. Keep this pair identical across
# 01_dataset_creation_lstm / 02_dataset_creation_flat / 06_dataset_creation_face_crops /
# 09_dataset_creation_cnn_lstm.
CLASS_NAMES = ["Not Drowsy", "Drowsy"]
NUM_CLASSES = len(CLASS_NAMES)
print(f"{NUM_CLASSES} classes: {CLASS_NAMES}")


In [ ]:
import os

# Define the subpaths for the Argus project structure
models_folder = f"{project_folder}/models"
dataset_folder = f"{project_folder}/dataset"
raw_videos_folder = f"{dataset_folder}/raw_videos_binary"
processed_folder = f"{dataset_folder}/dataset_processed"

if not os.path.exists(raw_videos_folder):
  raise Exception(f"The target directory could not be initialized: '{raw_videos_folder}'.")

# List of all directories that need initialize
folders_to_create = [
    models_folder,
    processed_folder,
]

# Create directories if they do not exist
for folder in folders_to_create:
    os.makedirs(folder, exist_ok=True)

print("Project folder structure successfully created and verified.")


Finally, check for uploaded subject directories. If no subjects exist, issue a warning to ensure data is present before proceeding.


In [ ]:
import glob

# Validation check: Ensure raw_videos folder contains video clips
# Search recursively for any .mp4 files within the subject folders
video_files = glob.glob(os.path.join(raw_videos_folder, "**/*.mp4"), recursive=True)
if len(video_files) == 0:
    raise Exception(f"No video clips found in the specified directory: '{raw_videos_folder}'.")
print(f"[SUCCESS] Raw video directory validated: Found {len(video_files)} clip(s) ready for processing.")

# Raw Video Dataset Exploratory Visualization / Histogram


This cell processes your raw video files to extract metadata, calculate exact video lengths, and organize them by subjects and drowsiness levels. Here is a breakdown of what the code does step-by-step:

* **Subject Mapping:**
  * Loops through all discovered `.mp4` paths and extracts the immediate parent folder name (e.g., `subject_01`) to track data ownership per person.
* **Duration Calculation (OpenCV):**
  * Opens each video to read its frames-per-second (`FPS`) and total frame count.
  * Divides total frames by FPS to get the exact clip duration in seconds, then accumulates it toward both the subject and the drowsiness level totals.
* **Level Parsing (Regex):**
  * Uses a regular expression pattern (`level_(\d+)`) to pull the drowsiness level integer from the filename. Unrecognized filenames are logged with a `-1` flag.
* **Summary Compilation:**
  * Aggregates final counts using `Counter` and prints a status message reporting total clips and subjects processed.

In [ ]:
import re
from collections import Counter
import cv2

if not video_files:
    raise Exception(f"No video clips found in '{raw_videos_folder}' to analyze. Please check your path or uploads.")

levels = []
subjects = []
level_durations = Counter()
subject_durations = Counter()

for file_path in video_files:
    filename = os.path.basename(file_path)

    # Extract subject ID from the parent directory name (e.g., subject_01)
    parent_dir = os.path.basename(os.path.dirname(file_path))
    subjects.append(parent_dir)

    # Calculate video duration using OpenCV
    cap = cv2.VideoCapture(file_path)
    duration_sec = 0
    if cap.isOpened():
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        if fps > 0:
            duration_sec = frame_count / fps
        cap.release()

    subject_durations[parent_dir] += duration_sec

    # Extract drowsiness level from the filename (e.g., level_1_clip_01.mp4 -> 1)
    match = re.search(r'level_(\d+)', filename, re.IGNORECASE)
    if match:
        level = int(match.group(1))
        levels.append(level)
        level_durations[level] += duration_sec
    else:
        levels.append(-1)

# Count frequencies
level_counts = Counter([lvl for lvl in levels if lvl != -1])
subject_counts = Counter(subjects)

print(f"[SUCCESS] Parsed {len(video_files)} total video clips across {len(subject_counts)} subjects.")

## Visualization - Drowsiness Clip Level Distribution

In [ ]:
import matplotlib.pyplot as plt

# Prepare data for Drowsiness Level distribution
sorted_levels = sorted(level_counts.keys())
level_vals = [level_counts[lvl] for lvl in sorted_levels]

plt.figure(figsize=(7, 5))
bars1 = plt.bar([f"L{lvl}\n{CLASS_NAMES[lvl - 1]}" if 1 <= lvl <= len(CLASS_NAMES) else f"Level {lvl}" for lvl in sorted_levels], level_vals, color='#3b82f6', edgecolor='#1e40af', width=0.6)

plt.title("Dataset Distribution by Class", fontsize=12, fontweight='bold')
plt.xlabel("Class", fontsize=11, fontweight='bold')
plt.ylabel("Number of Clips", fontsize=11, fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.7)

for bar in bars1:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, height + 0.1, str(height), ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## Visualization - Total Duration by Drowsiness Level

In [ ]:
import matplotlib.pyplot as plt

# Prepare data for Drowsiness Level duration distribution (converted to minutes)
sorted_levels = sorted(level_durations.keys())
level_vals_minutes = [level_durations[lvl] / 60.0 for lvl in sorted_levels]

plt.figure(figsize=(7, 5))
bars1 = plt.bar([f"L{lvl}\n{CLASS_NAMES[lvl - 1]}" if 1 <= lvl <= len(CLASS_NAMES) else f"Level {lvl}" for lvl in sorted_levels], level_vals_minutes, color='#3b82f6', edgecolor='#1e40af', width=0.6)

plt.title("Dataset Total Duration by Class", fontsize=12, fontweight='bold')
plt.xlabel("Class", fontsize=11, fontweight='bold')
plt.ylabel("Total Duration (Minutes)", fontsize=11, fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.7)

for bar in bars1:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, height + 0.1, f"{height:.1f}m", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## Visualization - Subject Distribution


In [ ]:
# Prepare data for Subject distribution
sorted_subjects = sorted(subject_counts.keys())
subject_vals = [subject_counts[subj] for subj in sorted_subjects]

plt.figure(figsize=(7, 5))
bars2 = plt.bar(sorted_subjects, subject_vals, color='#10b981', edgecolor='#047857', width=0.6)

plt.title("Dataset Distribution by Subject", fontsize=12, fontweight='bold')
plt.xlabel("Subject ID", fontsize=11, fontweight='bold')
plt.ylabel("Number of Clips", fontsize=11, fontweight='bold')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

for bar in bars2:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, height + 0.1, str(height), ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"[SUCCESS] Successfully rendered all exploratory distribution charts for {len(video_files)} clips.")

## Visualization - Total Duration by Subject

In [ ]:
import matplotlib.pyplot as plt

# Prepare data for Subject duration distribution (converted to minutes)
sorted_subjects = sorted(subject_durations.keys())
subject_vals_minutes = [subject_durations[subj] / 60.0 for subj in sorted_subjects]

plt.figure(figsize=(7, 5))
bars2 = plt.bar(sorted_subjects, subject_vals_minutes, color='#10b981', edgecolor='#047857', width=0.6)

plt.title("Dataset Total Duration by Subject", fontsize=12, fontweight='bold')
plt.xlabel("Subject ID", fontsize=11, fontweight='bold')
plt.ylabel("Total Duration (Minutes)", fontsize=11, fontweight='bold')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

for bar in bars2:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, height + 0.1, f"{height:.1f}m", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"[SUCCESS] Successfully rendered subject duration distribution chart.")

#  Media Pipe Model Setup & Feature Engineering & Dataset Assembly Pipeline Orchestration


## Dataset Assembly Pipeline Orchestration

This section turns raw `.mp4` clips into structured artifacts for LSTM training.

### The Processing Logic:
1. **Feature Extraction:** MediaPipe extracts landmarks and blendshapes at 10 FPS.
2. **Stream Generation:** The pipeline slices the continuous data into overlapping "streams" (1s–6s windows).
3. **Quality Gate:** The script checks index 6 (`ear_mar_valid`) of every frame in a window. If any frame is marked as invalid (face lost), the entire window is dropped.
4. **Indexing:** Metadata is generated to map every individual `.npy` stream back to its original clip and subject.

## Model Media Pipe Setup & Installing Dependencies



Before any feature extraction can run, a pretrained MediaPipe FaceLandmarker model bundle must exist inside models_folder. This section is idempotent and safe to re-run: it checks for an existing model first, and only downloads one if missing — so once it has run once for a given Google Drive, later runs (or other notebooks sharing the same Argus/models/ folder) reuse the cached file instead of re-downloading.

### Downloading and Setting Up the MediaPipe Model Bundle
1. Install Dependencies: Run the installation command to ensure `mediapipe` and `opencv-python` are available in your environment.

2. Define Directory & Path: Set your target directory (`models_folder`) and verify or create it.

3. Download and Move Model: Check if the pretrained `MediaPipe` `FaceLandmarker` model bundle exists, download it if missing, and move/save it into `models_folder`.


In [ ]:
!pip install mediapipe opencv-python

In [ ]:
# Verify that the model exist
!ls -l {models_folder}

In [ ]:
import urllib.request

media_pipe_url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task"
media_pipe_filename = "face_landmarker.task"
media_pipe_path = os.path.join(models_folder, media_pipe_filename)

# Download the model only if it doesn't already exist and catch any errors
if not os.path.exists(media_pipe_path):
    print(f"Downloading MediaPipe Face Landmarker model to {media_pipe_path}...")
    try:
        urllib.request.urlretrieve(media_pipe_url, media_pipe_path)
        print("Download complete.")
    except Exception as e:
        raise RuntimeError(f"Failed to download the MediaPipe model from {media_pipe_url}. Error: {e}")
else:
    print(f"Model already exists at {media_pipe_path}. Skipping download.")

### # MediaPipe Face Landmarker Quantization -- (Still Working on it)



This subsection implements the workflow for the **Argus** drowsiness detection project, focusing on downloading, extracting, and optimizing the MediaPipe Face Landmarker model for deployment.

1. Installing the AI quantizer library

In [ ]:
# !pip install -q ai-edge-quantizer

2. Extract the `.task` bundle (it is a ZIP archive)

In [ ]:
# import shutil
# import zipfile

# extract_dir = "extracted_task"
# if os.path.exists(extract_dir):
#     shutil.rmtree(extract_dir)   # clean up previous extraction

# with zipfile.ZipFile(media_pipe_path, 'r') as zip_ref:
#     zip_ref.extractall(extract_dir)

# print(f"Extracted contents to {extract_dir}/")
# !ls -lh {extract_dir}

3. Locate all TFLite models inside the extracted folder

In [ ]:
# # Build the full paths
# model_paths = {
#     "face_detector": os.path.join(extract_dir, "face_detector.tflite"),
#     "face_landmarks": os.path.join(extract_dir, "face_landmarks_detector.tflite"),
#     "face_blendshapes": os.path.join(extract_dir, "face_blendshapes.tflite"),
# }

# # Validate all exist before starting
# missing = [name for name, path in model_paths.items() if not os.path.exists(path)]
# if missing:
#     raise FileNotFoundError(
#         f"The following required TFLite models are missing from '{extract_dir}': {', '.join(missing)}"
#     )

4. Quantize the TFLite modeles using `ai_edge_quantizer` and replace them.

In [ ]:
# from ai_edge_quantizer import quantizer, recipe
# import shutil
# import os

# # Using the dynamic_wi8_afp32 recipe to achieve INT8 quantization.
# # This reduces weight precision to 8-bit integers while maintaining activations in float32,
# # which is the targeted optimization for the Argus project deployment.
# for name, path in model_paths.items():
#     print(f"Quantizing {name} to INT8 (Dynamic)...")
#     qt = quantizer.Quantizer(path)

#     # Load the dynamic INT8 quantization recipe
#     qt.load_quantization_recipe(recipe.dynamic_wi8_afp32())

#     # Define the temporary output path for the INT8 model
#     quantized_temp = f"{name}_dynamic_int8.tflite"
#     if os.path.exists(quantized_temp):
#         os.remove(quantized_temp)

#     # Execute the quantization and export
#     qt.quantize().export_model(quantized_temp)

#     # Replace the original file in the extracted folder with the INT8 version
#     shutil.move(quantized_temp, path)
#     print(f"  Successfully replaced {path} with dynamic INT8 version.")

6. Repackage the folder into a new `.task` file (ZIP with no compression)

In [ ]:
# new_task_path = os.path.join(models_folder, "face_landmarker_quantized.task")
# if os.path.exists(new_task_path):
#     os.remove(new_task_path)

# with zipfile.ZipFile(new_task_path, 'w', zipfile.ZIP_STORED) as zipf:
#     for root, dirs, files in os.walk(extract_dir):
#         for file in files:
#             full_path = os.path.join(root, file)
#             arcname = os.path.relpath(full_path, extract_dir)
#             zipf.write(full_path, arcname)

# print(f"✅ Quantized .task file created: {new_task_path}")
# print(f"   Size: {os.path.getsize(new_task_path):,} bytes")

7. Quick validation – load the quantized model with MediaPipe

In [ ]:
# from mediapipe.tasks import python
# from mediapipe.tasks.python import vision

# base_options = python.BaseOptions(model_asset_path=new_task_path)
# options = vision.FaceLandmarkerOptions(
#     base_options=base_options,
#     running_mode=vision.RunningMode.IMAGE,
#     num_faces=1,
#     min_face_detection_confidence=0.5
# )
# detector = vision.FaceLandmarker.create_from_options(options)

# print("✅ Quantized model loaded successfully!")

### Test MediaPipe Face Landmarker on a Random Frame

First, let's import the necessary libraries and set up the `FaceLandmarker` for image mode, as we'll be processing a single static image.

In [ ]:
import random
import cv2
import os
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

# Fix: Explicitly define base_options using the path resolved in the download step
base_options = mp_python.BaseOptions(model_asset_path=os.path.abspath(media_pipe_path))

# Creating new options for IMAGE running mode
image_landmarker_options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=True,
)

# Create the FaceLandmarker detector instance
detector = vision.FaceLandmarker.create_from_options(image_landmarker_options)

print("✅ MediaPipe FaceLandmarker detector initialized for IMAGE mode.")

Next, we'll randomly select one of your video files and then pick a random frame from it. This frame will be used to test the Face Landmarker.

In [ ]:
# Randomly select a video file
selected_video_path = random.choice(video_files)
print(f"Selected video for test: {selected_video_path}")

# Open the video file
cap = cv2.VideoCapture(selected_video_path)
if not cap.isOpened():
    raise IOError(f"Cannot open video file: {selected_video_path}")

# Get total frame count
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
if frame_count == 0:
    raise ValueError(f"Video file {selected_video_path} has no frames.")

# Select a random frame index
random_frame_idx = random.randint(0, frame_count - 1)
print(f"Selected random frame index: {random_frame_idx}")

# Set the video capture to the random frame
cap.set(cv2.CAP_PROP_POS_FRAMES, random_frame_idx)

# Read the frame
ret, bgr_frame = cap.read()
cap.release()

if not ret:
    raise Exception(f"Failed to read frame {random_frame_idx} from {selected_video_path}")

# Display the raw selected frame
plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)) # Convert BGR to RGB for matplotlib
plt.title(f"Random Frame {random_frame_idx} from {os.path.basename(selected_video_path)}")
plt.axis('off')
plt.show()

print("✅ Random frame selected and displayed.")

Now we'll run the `FaceLandmarker` detector on the selected frame. The `detector` will identify faces and extract facial landmarks, blendshapes, and the head pose transformation matrix.

In [ ]:
# Convert the OpenCV BGR frame to RGB and then to a MediaPipe Image object
rgb_frame = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

# Perform face landmark detection
detection_result = detector.detect(mp_image)

if detection_result.face_landmarks:
    print(f"✅ Face(s) detected: {len(detection_result.face_landmarks)}")
    # The first face's blendshapes can be accessed via: detection_result.face_blendshapes[0]
    # And transformation matrix via: detection_result.facial_transformation_matrixes[0]
else:
    print("⚠️ No face detected in the selected frame.")

Finally, let's visualize the detected landmarks on the original frame. We'll use MediaPipe's drawing utilities for this.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks.python import vision

# Use the existing detector (from your previous cells)
# detector is already created

# Convert frame and detect (assuming detection_result is already available)
# If not, run detection again:
# rgb_frame = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
# mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
# detection_result = detector.detect(mp_image)

# Make a copy of the frame to annotate
annotated_frame = bgr_frame.copy()
height, width, _ = annotated_frame.shape

if detection_result.face_landmarks:
    # Get the first face's landmarks
    landmarks = detection_result.face_landmarks[0]  # list of NormalizedLandmark

    # Draw each landmark as a small circle
    for lm in landmarks:
        x = int(lm.x * width)
        y = int(lm.y * height)
        cv2.circle(annotated_frame, (x, y), 1, (0, 255, 0), -1)   # green dots

    # Optionally, draw connections (tesselation) – you can import the connection list
    # from the old API without needing the full solutions module:
    try:
        from mediapipe.solutions.face_mesh_connections import FACEMESH_TESSELATION
    except ImportError:
        # Fallback: define a minimal set of connections (or skip)
        FACEMESH_TESSELATION = []

    # Draw connections as lines (if available)
    for connection in FACEMESH_TESSELATION:
        idx1, idx2 = connection
        # Ensure indices are valid
        if idx1 < len(landmarks) and idx2 < len(landmarks):
            x1 = int(landmarks[idx1].x * width)
            y1 = int(landmarks[idx1].y * height)
            x2 = int(landmarks[idx2].x * width)
            y2 = int(landmarks[idx2].y * height)
            cv2.line(annotated_frame, (x1, y1), (x2, y2), (0, 255, 0), 1)

    # Display the annotated frame
    plt.figure(figsize=(10, 8))
    plt.imshow(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB))
    plt.title("Frame with Detected Face Landmarks")
    plt.axis('off')
    plt.show()
    print("✅ Annotated frame displayed.")
else:
    print("No face landmarks to draw.")

## Feature Extraction per frame

1. FaceLandmaker initialization & Landmark Index Constnats
2. Geometric Ratio Tensor Layer For Functions (EAR, MAR, Head Pose)

### 1. FaceLandmarker Initialization & Landmark Index Constants



To ensure temporal consistency and avoid state-tracking errors across different video files, we configure the `FaceLandmarker` options to use `VIDEO` running mode. While the configuration is defined globally in `face_landmarker_options`, the **FeatureExtractionPipeline** now instantiates a fresh detector for every video processed. This resets the internal MediaPipe timestamp clock, preventing errors related to non-monotonic timestamps when moving from the end of one clip to the start of another.

Key configuration highlights:

*   **Blendshapes & Transformation Matrix:** Both are enabled to extract 52 facial action units and head pose (pitch/yaw/roll) simultaneously.
*   **Landmark Indices:** `left_eye_ear_idx`, `right_eye_ear_idx`, and `mouth_mar_idx` use standard MediaPipe FaceMesh topology (6-point eyes, 4-point mouth) which remains stable across all subjects.
*   **Sampling Strategy:** `sampling_fps` (defaulting to 10) ensures the training data matches the temporal distribution achievable on the target Raspberry Pi 5 hardware.

This setup reuses the `media_pipe_path` resolved in the previous download step.

In [ ]:
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

# --- FaceLandmarker configuration (media_pipe_path already resolved by the download step) ---
base_options = mp_python.BaseOptions(model_asset_path=os.path.abspath(media_pipe_path))
face_landmarker_options = vision.FaceLandmarkerOptions(
    base_options = base_options,
    running_mode = vision.RunningMode.VIDEO,
    num_faces = 1,
    min_face_detection_confidence = 0.5,
    min_face_presence_confidence = 0.5,
    min_tracking_confidence = 0.5,
    output_face_blendshapes = True,
    output_facial_transformation_matrixes = True,
)

print("FaceLandmarker configuration and constants ready.")

### 2. Geometric Ratio Tensor Layer For Functions (EAR, MAR, Head Pose)

Here we define the TensorFlow layer that we will use both during dataset building (for consistency) and later when we assemble the LSTM model.

The geometric helper functions needed for computing (`EAR`, `MAR`, `Head Pose`)

To enable export to a single `.tflite` file — and to keep feature extraction identical between dataset creation and inference — we now implement an exact TensorFlow‑native counterpart: **`GeometricRatioFeatureLayer`**. This layer performs the same geometric calculations using `tf` operations.

In [ ]:
import tensorflow as tf
import numpy as np

@tf.keras.utils.register_keras_serializable(package="Argus")
class GeometricRatioFeatureLayer(tf.keras.layers.Layer):
    """
    TensorFlow-native counterpart to compute_ear, compute_mar, and rotation_matrix_to_euler.
    Inputs:
        landmarks_xy : (batch, 478, 2)   -- normalized (x, y) coordinates
        rotation_matrix : (batch, 3, 3)  -- top‑left 3×3 block of the facial transformation matrix
    Returns:
        (batch, 7) : [EAR_left, EAR_right, MAR, pitch, yaw, roll, ear_mar_valid]
    """
    def __init__(self, pose_validity_threshold_deg=20.0, **kwargs):
        super().__init__(**kwargs)
        self.pose_validity_threshold_deg = pose_validity_threshold_deg
        self.left_eye_idx  = tf.constant([33, 160, 158, 133, 153, 144], dtype=tf.int32)
        self.right_eye_idx = tf.constant([362, 385, 387, 263, 373, 380], dtype=tf.int32)
        self.mouth_idx     = tf.constant([61, 291, 13, 14], dtype=tf.int32)
        self.blendshape_names = [
            "browDownLeft", "browDownRight", "browInnerUp", "browOuterUpLeft", "browOuterUpRight",
            "cheekPuff", "cheekSquintLeft", "cheekSquintRight", "eyeBlinkLeft", "eyeBlinkRight",
            "eyeLookDownLeft", "eyeLookDownRight", "eyeLookInLeft", "eyeLookInRight",
            "eyeLookOutLeft", "eyeLookOutRight", "eyeLookUpLeft", "eyeLookUpRight",
            "eyeSquintLeft", "eyeSquintRight", "eyeWideLeft", "eyeWideRight",
            "jawForward", "jawLeft", "jawOpen", "jawRight", "mouthClose", "mouthDimpleLeft",
            "mouthDimpleRight", "mouthFrownLeft", "mouthFrownRight", "mouthFunnel",
            "mouthLeft", "mouthLowerDownLeft", "mouthLowerDownRight", "mouthPressLeft",
            "mouthPressRight", "mouthPucker", "mouthRight", "mouthRollLower",
            "mouthRollUpper", "mouthShrugLower", "mouthShrugUpper", "mouthSmileLeft",
            "mouthSmileRight", "mouthStretchLeft", "mouthStretchRight", "mouthUpperUpLeft",
            "mouthUpperUpRight", "noseSneerLeft", "noseSneerRight"
        ]

    def get_config(self):
        config = super().get_config()
        config.update({"pose_validity_threshold_deg": self.pose_validity_threshold_deg})
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

    @staticmethod
    def _dist(points, i, j):
        a = points[..., i, :]
        b = points[..., j, :]
        return tf.norm(a - b, axis=-1)

    def _ear(self, landmarks, idx):
        p = tf.gather(landmarks, idx, axis=-2)
        vertical = self._dist(p, 1, 5) + self._dist(p, 2, 4)
        horizontal = 2.0 * self._dist(p, 0, 3)
        return tf.math.divide_no_nan(vertical, horizontal)

    def _mar(self, landmarks, idx):
        p = tf.gather(landmarks, idx, axis=-2)
        vertical = self._dist(p, 2, 3)
        horizontal = self._dist(p, 0, 1)
        return tf.math.divide_no_nan(vertical, horizontal)

    @staticmethod
    def _rotation_matrix_to_euler(R):
        r00, r10, r20 = R[..., 0, 0], R[..., 1, 0], R[..., 2, 0]
        r21, r22 = R[..., 2, 1], R[..., 2, 2]
        r11, r12 = R[..., 1, 1], R[..., 1, 2]
        sy = tf.sqrt(r00**2 + r10**2)
        singular = sy < 1e-6
        pitch = tf.where(singular, tf.atan2(-r12, r11), tf.atan2(r21, r22))
        yaw = tf.atan2(-r20, sy)
        roll = tf.where(singular, tf.zeros_like(r10), tf.atan2(r10, r00))
        return pitch * (180.0 / np.pi), yaw * (180.0 / np.pi), roll * (180.0 / np.pi)

    def call(self, landmarks_xy, rotation_matrix):
        ear_left = self._ear(landmarks_xy, self.left_eye_idx)
        ear_right = self._ear(landmarks_xy, self.right_eye_idx)
        mar = self._mar(landmarks_xy, self.mouth_idx)
        pitch, yaw, roll = self._rotation_matrix_to_euler(rotation_matrix)
        ear_mar_valid = tf.cast(tf.logical_and(tf.abs(yaw) < self.pose_validity_threshold_deg, tf.abs(pitch) < self.pose_validity_threshold_deg), tf.float32)
        return tf.stack([ear_left, ear_right, mar, pitch, yaw, roll, ear_mar_valid], axis=-1)

## Pipeline Orchestration: Connecting MediaPipe, Ratio Layer, and Video Processing

### 1. Pipeline Configuration Constants





This subsection defines the global parameters that govern the feature extraction and windowing logic. Adjusting these values allows you to experiment with different temporal resolutions and window sizes without modifying the core pipeline code.

In [ ]:
# --- Pipeline Configuration Constants ---
sampling_fps = 10                     # Target extraction rate (frames per second)
min_context_sec = 1                  # Minimum window size
max_context_sec = 6.0                # Maximum window size
pose_validity_threshold_deg = 20.0    # Pitch/Yaw range for valid EAR/MAR

# Dynamic window configuration: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
# This creates the array of durations for the LSTM streams
window_configs = [float(x) for x in range(int(min_context_sec), int(max_context_sec) + 1)]
stride_sec = 1.0  # Slide the window by 1 second steps

# Every window, regardless of its real duration, is zero-*pre*-padded up to this many timesteps
# before being written to lstm_windows.csv (see the "From thousands of .npy files to one padded
# CSV" note above). This was originally 120 (double max_context_sec * sampling_fps = 60, kept
# as unused headroom for longer-context experiments later) but was brought down to exactly
# max_context_sec * sampling_fps = 60 -- zero headroom, but every window actually generated
# (1-6s) still fits -- after a full extraction run at 120 OOM-crashed the Colab kernel. Peak
# RAM here scales directly with MAX_TIMESTEPS: the "Video Processing Loop" cell flattens every
# window into a (6 metadata + MAX_TIMESTEPS * num_features)-wide row and holds ALL windows from
# ALL videos in one Python list before a single pd.DataFrame(rows) call builds the whole
# dataset at once -- at 120 that is 6,966 columns/row across potentially tens of thousands of
# windows, all held in memory simultaneously. Halving MAX_TIMESTEPS roughly halves that peak.
# This constant must match 03_model_training_lstm.ipynb's MAX_TIMESTEPS exactly, since that
# notebook reshapes CSV rows straight back into (MAX_TIMESTEPS, num_features) with no further
# padding.
MAX_TIMESTEPS = 60

# --- Raw level (from filenames) -> validated class label ---
# Filenames already encode the final class (level_1 = Not Drowsy, level_2 = Drowsy), so
# map_level does no remapping -- it's a validation pass-through, kept as a function (not
# inlined int(...)) so a malformed level fails loudly at one place. Keep identical to the
# copies in 02_dataset_creation_flat.ipynb / 06_dataset_creation_face_crops.ipynb.
def map_level(raw_level: int) -> int:
    if raw_level not in (1, 2):
        raise ValueError(f"Unexpected level {raw_level} in filename -- expected 1 (Not Drowsy) or 2 (Drowsy).")
    return raw_level

print(f"✅ Pipeline constants initialized.")
print(f"   Sampling: {sampling_fps} FPS")
print(f"   Window Streams to generate: {window_configs} seconds")
print(f"   MAX_TIMESTEPS (fixed pad length): {MAX_TIMESTEPS}")

# --- Fixed landmark index sets for EAR/MAR ---
left_eye_ear_idx  = [33, 160, 158, 133, 153, 144]
right_eye_ear_idx = [362, 385, 387, 263, 373, 380]
mouth_mar_idx     = [61, 291, 13, 14]


### 2. Pipeline Orchestration Definition

Now we define the core orchestration logic that processes a raw video and produces the feature matrix.

**Components:**
- **MediaPipe detector** – extracts landmarks, blendshapes, and the 3×3 rotation matrix per frame.
- **`GeometricRatioFeatureLayer`** – computes EAR, MAR, and head‑pose angles (pitch, yaw, roll) from the raw landmarks and rotation matrix.
- **Video reader** – uses OpenCV to sample frames at the target `sampling_fps`.

The pipeline returns a NumPy array of shape `(num_frames, 59)` where each row consists of:
- 7 base features: `[EAR_left, EAR_right, MAR, pitch, yaw, roll, ear_mar_valid]`
- 52 blendshape scores (MediaPipe’s ARKit‑style coefficients)

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
from mediapipe.tasks.python import vision
import mediapipe as mp
from tqdm.auto import tqdm
import os

class FeatureExtractionPipeline:
    def __init__(self, face_landmarker_options: vision.FaceLandmarkerOptions,
                 ratio_layer: tf.keras.layers.Layer,
                 sampling_fps: int = 10):
        self.face_landmarker_options = face_landmarker_options
        self.ratio_layer = ratio_layer
        self.sampling_fps = sampling_fps

    def process_video(self, video_path: str):
        # Create a fresh detector per video to avoid timestamp issues
        detector_local = vision.FaceLandmarker.create_from_options(self.face_landmarker_options)
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            detector_local.close()
            raise IOError(f"Cannot open video: {video_path}")

        src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_stride = max(1, round(src_fps / self.sampling_fps))

        rows = []
        dropped_frames = 0
        frame_idx = 0

        pbar = tqdm(total=total_frames, desc=f"Processing {os.path.basename(video_path)[:20]}...", leave=False)

        while True:
            ret, frame_bgr = cap.read()
            if not ret:
                break

            if frame_idx % frame_stride == 0:
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
                timestamp_ms = int(frame_idx * (1000.0 / src_fps))

                result = detector_local.detect_for_video(mp_image, timestamp_ms)

                if result.face_landmarks:
                    lm = result.face_landmarks[0]
                    landmarks_xy = tf.constant([[p.x, p.y] for p in lm], dtype=tf.float32)
                    R = np.array(result.facial_transformation_matrixes[0])[:3, :3] if result.facial_transformation_matrixes else np.eye(3, dtype=np.float32)
                    R_tf = tf.constant(R, dtype=tf.float32)

                    # Compute EAR, MAR, Pose using the TF layer
                    ratios = self.ratio_layer(landmarks_xy[tf.newaxis, ...], R_tf[tf.newaxis, ...]).numpy()[0]

                    # Extract 52 blendshapes
                    bs_dict = {b.category_name: b.score for b in result.face_blendshapes[0]} if result.face_blendshapes else {}
                    bs_scores = [bs_dict.get(name, 0.0) for name in self.ratio_layer.blendshape_names]

                    row = np.concatenate([ratios, bs_scores]).astype(np.float32)
                    rows.append(row)
                else:
                    dropped_frames += 1
                    # Append a dummy row where validity (index 6) is 0.0
                    # The label mapping happens in the loop cell based on the filename.
                    num_features = 7 + len(self.ratio_layer.blendshape_names)
                    dummy = np.zeros(num_features, dtype=np.float32)
                    rows.append(dummy)

            frame_idx += 1
            pbar.update(1)

        pbar.close()
        cap.release()
        detector_local.close()

        return (np.array(rows, dtype=np.float32), dropped_frames) if rows else (None, 0)

### 3. Pipeline Orchestration Run & Dataset Creation

This section orchestrates the feature extraction pipeline, processing each raw video clip and appending its padded windows as rows of `lstm_windows.csv`. This process is crucial for transforming raw video data into a structured dataset ready for model training.


### Initial Setup and Blendshape Names

First, we ensure that necessary variables like `metadata_path` are defined. A critical step is defining the `blendshape_names` list, which enumerates all 52 ARKit-style blendshape scores that MediaPipe can output. This list ensures a consistent order for our feature vectors, even if certain blendshapes are not present in a given frame.

In [ ]:
# --- Ensure required variables are defined ---
# (These should already exist, but we double-check)
lstm_windows_csv_path = os.path.join(processed_folder, "lstm_windows.csv")

# --- Ensure the ratio layer has the blendshape names (if not already defined) ---
if not hasattr(GeometricRatioFeatureLayer, 'blendshape_names'):
    GeometricRatioFeatureLayer.blendshape_names = [
        "browDownLeft", "browDownRight", "browInnerUp", "browOuterUpLeft", "browOuterUpRight",
        "cheekPuff", "cheekSquintLeft", "cheekSquintRight",
        "eyeBlinkLeft", "eyeBlinkRight",
        "eyeLookDownLeft", "eyeLookDownRight", "eyeLookInLeft", "eyeLookInRight",
        "eyeLookOutLeft", "eyeLookOutRight", "eyeLookUpLeft", "eyeLookUpRight",
        "eyeSquintLeft", "eyeSquintRight", "eyeWideLeft", "eyeWideRight",
        "jawForward", "jawLeft", "jawOpen", "jawRight",
        "mouthClose", "mouthDimpleLeft", "mouthDimpleRight", "mouthFrownLeft", "mouthFrownRight",
        "mouthFunnel", "mouthLeft", "mouthLowerDownLeft", "mouthLowerDownRight",
        "mouthPressLeft", "mouthPressRight", "mouthPucker", "mouthRight",
        "mouthRollLower", "mouthRollUpper", "mouthShrugLower", "mouthShrugUpper",
        "mouthSmileLeft", "mouthSmileRight", "mouthStretchLeft", "mouthStretchRight",
        "mouthUpperUpLeft", "mouthUpperUpRight", "noseSneerLeft", "noseSneerRight",
    ]

# Real per-frame feature count: 7 geometric ratios + len(blendshape_names) (51, not the
# documented 52 -- MediaPipe's blendshape_names list is missing "_neutral"). Derived here
# rather than hardcoded so it can't silently drift from the actual data again.
num_features = 7 + len(GeometricRatioFeatureLayer.blendshape_names)
print(f"num_features = {num_features}")


### Instantiate Components and Dataset Reset Option

Here, we initialize the `FaceLandmarker` detector, the `GeometricRatioFeatureLayer` (which calculates EAR, MAR, and head pose from landmarks), and the `FeatureExtractionPipeline`.

Crucially, a `reset_dataset` flag is introduced. If set to `True`, this will delete the previously generated `lstm_windows.csv`, allowing for a complete rebuild of the dataset. This is useful when you want to re-run the entire pipeline or update feature extraction logic.


In [ ]:
# --- Instantiate components ---
# The global 'detector' variable for VIDEO mode is removed here.
# FeatureExtractionPipeline will now create its own detector instance per video.
ratio_layer = GeometricRatioFeatureLayer(pose_validity_threshold_deg=pose_validity_threshold_deg)
pipeline = FeatureExtractionPipeline(face_landmarker_options, ratio_layer, sampling_fps=sampling_fps)

In [ ]:
import os

# --- Safety flag: set to True to delete existing dataset and rebuild from scratch ---
reset_dataset = True  # Change to True when you want to rebuild

# --- Reset if requested ---
if reset_dataset:
    if os.path.exists(lstm_windows_csv_path):
        os.remove(lstm_windows_csv_path)
        print(f"🧹 Existing windowed dataset removed: {lstm_windows_csv_path}")
    print("✨ Ready for fresh extraction.")


### Video Processing Loop

This is the core of the pipeline where each video file is processed. For each video:

1.  **Metadata Extraction:** The subject and level are parsed from the file path and name, then validated via `map_level` — `raw_videos_binary/` already encodes the final class (`level_1` / `level_2`) in the filename, so there is no remapping (see "Pipeline Configuration Constants").
2.  **Feature Extraction:** The `pipeline.process_video` method is called to extract per-frame features from the video.
3.  **Handling Skipped Videos:** If no face is detected in any frame or processing fails, the video is skipped and logged.
4.  **Windowing & Padding:** Each valid (no dropped frames inside it) sliding window is zero-*pre*-padded up to `MAX_TIMESTEPS` and flattened into one row.
5.  **Collecting Rows:** Metadata (subject, level, window duration, dropped-frame count) plus the flattened, padded feature values are stored as one dict per window in a list (`rows`), later written straight to `lstm_windows.csv`.


In [ ]:
import re
import numpy as np
import os
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

# --- Worker-local pipeline (built once per worker process, never shipped through pickle) ------
# ProcessPoolExecutor.submit() always pickles its call arguments to ship them to worker
# processes -- this is true regardless of the fork/spawn start method. The original version of
# this cell passed the already-built `pipeline` object (a FeatureExtractionPipeline holding a
# live GeometricRatioFeatureLayer -- a TF Keras layer with tf.constant tensor attributes -- and
# a MediaPipe FaceLandmarkerOptions/BaseOptions wrapping native state) straight into
# executor.submit(). Unpickling a live TensorFlow/MediaPipe object inside a different process
# than the one that created it is a native-level operation that can crash the worker outright
# (a segfault, not a catchable Python exception) -- which is exactly what
# "BrokenProcessPool: A process in the process pool was terminated abruptly" means, and why it
# kept happening no matter how many workers were used.
#
# The fix: never pass a live TF/MediaPipe object across the process boundary. Instead, pass only
# plain picklable primitives (paths, floats, dicts of strings/ints) and have each worker build
# its own GeometricRatioFeatureLayer + FaceLandmarkerOptions + FeatureExtractionPipeline exactly
# once, via ProcessPoolExecutor's `initializer` hook, entirely within that worker's own process.
_worker_pipeline = None

def _init_worker(media_pipe_path, pose_validity_threshold_deg_, blendshape_names, sampling_fps_):
    global _worker_pipeline
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python import vision

    if not hasattr(GeometricRatioFeatureLayer, "blendshape_names"):
        GeometricRatioFeatureLayer.blendshape_names = blendshape_names
    worker_ratio_layer = GeometricRatioFeatureLayer(pose_validity_threshold_deg=pose_validity_threshold_deg_)

    worker_base_options = mp_python.BaseOptions(model_asset_path=os.path.abspath(media_pipe_path))
    worker_face_landmarker_options = vision.FaceLandmarkerOptions(
        base_options=worker_base_options,
        running_mode=vision.RunningMode.VIDEO,
        num_faces=1,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_face_blendshapes=True,
        output_facial_transformation_matrixes=True,
    )
    _worker_pipeline = FeatureExtractionPipeline(worker_face_landmarker_options, worker_ratio_layer, sampling_fps=sampling_fps_)


def process_single_video(video_path, window_configs, sampling_fps, stride_sec, max_timesteps):
    """Helper function to process a single video for parallel execution.

    Returns a list of flat row-dicts (metadata columns + one column per (timestep, feature)
    pair), each already zero-pre-padded to max_timesteps -- no separate .npy files.

    Uses the worker-local `_worker_pipeline` built once per process by `_init_worker` above,
    rather than receiving a pipeline object as an argument -- see the note above `_init_worker`
    for why that distinction is what fixes BrokenProcessPool crashes.
    """
    filename = os.path.basename(video_path)
    subject = os.path.basename(os.path.dirname(video_path))

    level_match = re.search(r'level_(\d+)', filename, re.IGNORECASE)
    if not level_match:
        return None, (filename, "Missing 'level_' prefix")

    raw_level = int(level_match.group(1))
    try:
        level = map_level(raw_level)   # module-level global; inherited by the pool workers on fork
    except ValueError as e:
        return None, (filename, str(e))

    try:
        result = _worker_pipeline.process_video(video_path)
    except Exception as e:
        return None, (filename, f"process_video raised: {e!r}")

    if result is None or result[0] is None:
        return None, (filename, "Face detection failure")

    full_features, drop_count = result[0], result[1]
    num_feat = full_features.shape[1]
    local_rows = []

    for win_sec in window_configs:
        win_size = int(win_sec * sampling_fps)
        stride = int(stride_sec * sampling_fps)

        for start in range(0, len(full_features) - win_size + 1, stride):
            end = start + win_size
            window_data = full_features[start:end]

            if np.any(window_data[:, 6] == 0.0):
                continue

            # Zero-*pre*-pad up to max_timesteps (zeros first, real frames last) -- must match
            # the deployment buffer's fill order, see the notebook-level note above.
            pad_amount = max_timesteps - win_size
            if pad_amount < 0:
                raise ValueError(
                    f"window of {win_size} frames exceeds MAX_TIMESTEPS={max_timesteps}; "
                    "increase MAX_TIMESTEPS or shrink max_context_sec."
                )
            padded = np.pad(window_data, ((pad_amount, 0), (0, 0)), mode="constant")

            row = {
                "subject": subject,
                "level": level,
                "parent_video": filename,
                "window_duration_sec": win_sec,
                "n_real_frames": win_size,
                "dropped_frames_in_video": drop_count,
            }
            # Flatten (max_timesteps, num_feat) -> one column per (t, feature) pair.
            flat = padded.astype(np.float32).reshape(-1)
            row.update({f"t{t:03d}_f{f:02d}": flat[t * num_feat + f]
                        for t in range(max_timesteps) for f in range(num_feat)})
            local_rows.append(row)

    return local_rows, None

if reset_dataset:
    rows = []
    skipped = []

    # Start small (1-2 workers) to confirm this runs clean before scaling up -- now that the
    # crash is fixed, the limiting factor is genuinely RAM/CPU count, so tune this against
    # whatever os.cpu_count() reports on your chosen runtime.
    max_workers = 2

    print(f"🚀 Starting parallel extraction with {max_workers} workers...")

    with ProcessPoolExecutor(
        max_workers=max_workers,
        initializer=_init_worker,
        initargs=(media_pipe_path, pose_validity_threshold_deg, GeometricRatioFeatureLayer.blendshape_names, sampling_fps),
    ) as executor:
        futures = [
            executor.submit(
                process_single_video,
                vp, window_configs, sampling_fps, stride_sec, MAX_TIMESTEPS,
            ) for vp in video_files
        ]

        main_pbar = tqdm(as_completed(futures), total=len(video_files), desc="Parallel Progress")

        for future in main_pbar:
            local_rows, error = future.result()
            if error:
                skipped.append(error)
            if local_rows:
                rows.extend(local_rows)

    print(f"\nParallel Extraction Finished. Generated {len(rows)} windows.")
    if skipped:
        print(f"Skipped {len(skipped)} files.")
else:
    print("\nreset_dataset is False. Skipping parallel processing.")


### Writing the Windowed CSV and Cleanup

After processing all videos, the collected rows (metadata columns + flattened, padded features)
are written to `lstm_windows.csv` in one shot via `pandas`, since the flattened feature columns
are generated dynamically (`t{timestep}_f{feature}`) rather than a fixed, hand-maintained
fieldnames list.

Finally, the `detector.close()` method is called to release any resources held by the MediaPipe
FaceLandmarker.


In [ ]:
# --- Write the windowed dataset CSV ---
import pandas as pd

if reset_dataset:
    df_lstm_windows = pd.DataFrame(rows)
    df_lstm_windows.to_csv(lstm_windows_csv_path, index=False)

    print(f"\n✅ Dataset built: {len(rows)} padded windows written to {lstm_windows_csv_path}")
    print(f"   CSV shape: {df_lstm_windows.shape} (columns = metadata + {MAX_TIMESTEPS} x {num_features} flattened features)")
    if skipped:
        print(f"⚠️  Skipped {len(skipped)} clips:")
        for name, reason in skipped:
            print(f"   - {name}: {reason}")

# Clean up detector
detector.close()


### Dataset Sanity Check
Let's verify that `lstm_windows.csv` contains every class (`1..NUM_CLASSES`), check the distribution of generated windows, and confirm a row can be reshaped back into `(MAX_TIMESTEPS, num_features)` without surprises.


In [ ]:
import pandas as pd
import numpy as np

# Load the dataset we just wrote
df_lstm_windows = pd.read_csv(lstm_windows_csv_path)

meta_cols = ["subject", "level", "parent_video", "window_duration_sec", "n_real_frames", "dropped_frames_in_video"]
feature_cols = [c for c in df_lstm_windows.columns if c not in meta_cols]

# Check unique levels and counts
level_counts = df_lstm_windows['level'].value_counts().sort_index()

print("--- Dataset Sanity Check ---")
print(f"Total windows generated: {len(df_lstm_windows)}")
print(f"Feature columns: {len(feature_cols)} (expected {MAX_TIMESTEPS} x {num_features} = {MAX_TIMESTEPS * num_features})")
print("\nWindow counts per level:")
for lvl, count in level_counts.items():
    name = CLASS_NAMES[lvl - 1] if 1 <= lvl <= len(CLASS_NAMES) else "?"
    print(f"  Level {lvl} ({name}): {count} windows")

# Quick validation
missing_levels = [l for l in range(1, NUM_CLASSES + 1) if l not in level_counts.index]
if not missing_levels:
    print(f"\n✅ Success: all {NUM_CLASSES} classes are present in the dataset.")
else:
    print(f"\n⚠️ Warning: Missing levels {missing_levels} in the processed data.")

# Round-trip check: reshape one row back into (MAX_TIMESTEPS, num_features) and confirm the
# real (non-padded) tail matches n_real_frames.
sample_row = df_lstm_windows.iloc[0]
sample_seq = sample_row[feature_cols].to_numpy(dtype=np.float32).reshape(MAX_TIMESTEPS, num_features)
n_real = int(sample_row["n_real_frames"])
pad_rows_are_zero = np.allclose(sample_seq[: MAX_TIMESTEPS - n_real], 0.0)
print(f"\nRound-trip check on row 0: reshaped to {sample_seq.shape}, "
      f"leading {MAX_TIMESTEPS - n_real} padding rows all-zero: {pad_rows_are_zero}")

display(df_lstm_windows[meta_cols].head())
